# Sesión 6 · Notebook 3 — Optimización de consultas (PromQL / KQL)

Una consulta lenta en un dashboard que se refresca cada 10 segundos es una
consulta lenta 8.640 veces al día.

Ideas clave:

- Agregar **temprano** con `sum by (...)` para reducir el número de series.
- No pedir ventanas de tiempo más grandes de lo necesario.
- Precalcular con **recording rules** lo que sea caro y frecuente.
- En Kibana: acotar el rango y filtrar por campos `keyword`.

## 1) Reducir cardinalidad en PromQL

**Cara** — devuelve una serie por cada combinación de etiquetas:

```promql
rate(orderflow_orders_processed_total[5m])
```

**Barata** — agrega a lo que realmente vas a graficar:

```promql
sum by (service) (rate(orderflow_orders_processed_total[5m]))
```

Cuantas menos series devuelva, menos trabajo hacen Prometheus, la red y Grafana.

In [ ]:
import requests

PROM = "http://localhost:9090"


def n_series(expr):
    r = requests.get(f"{PROM}/api/v1/query", params={"query": expr}, timeout=10)
    return len(r.json()["data"]["result"])


sin_agregar = "rate(orderflow_orders_processed_total[5m])"
agregada = "sum by (service) (rate(orderflow_orders_processed_total[5m]))"

print("sin agregar :", n_series(sin_agregar))
print("agregada    :", n_series(agregada))

## 2) El coste real de una etiqueta

Esta es la cuenta que conviene saber hacer antes de añadir una etiqueta nueva a
una métrica: el número de series es el **producto** de los valores posibles de
todas sus etiquetas.

In [ ]:
def series_totales(**etiquetas):
    """Número de series que genera una métrica, dado el número de
    valores posibles de cada etiqueta."""
    total = 1
    for valores in etiquetas.values():
        total *= valores
    return total


print("region(3) x status(2)                :", series_totales(region=3, status=2))
print("region(3) x status(2) x customer(5k) :", series_totales(region=3, status=2, customer=5000))
print("... y si añades order_id (1/seg, 1 dia):",
      series_totales(region=3, status=2, customer=5000, order_id=86400))

Ese último número es la razón por la que `order_id` **nunca** debe ser una
etiqueta. Cada serie ocupa memoria en Prometheus de forma permanente, aunque
solo reciba un dato.

La regla práctica: una etiqueta es aceptable si sus valores posibles son pocos,
conocidos y estables. `region` sí. `customer_id` no. `order_id`, jamás.

Lo que necesita cardinalidad alta va en los **logs**, no en las métricas. Por eso
el `order_id` aparece en Elasticsearch y no en Prometheus.

## 3) Recording rules

Una *recording rule* calcula una expresión cara cada cierto intervalo y guarda el
resultado como una métrica nueva. El dashboard consulta el resultado ya hecho.

Archivo `prometheus/recording_rules.yml`:

```yaml
groups:
  - name: orderflow_recording
    interval: 30s
    rules:
      - record: job:orderflow_error_ratio:5m
        expr: >-
          sum(rate(orderflow_orders_failed_total[5m]))
          / clamp_min(sum(rate(orderflow_orders_processed_total[5m]))
          + sum(rate(orderflow_orders_failed_total[5m])), 0.001)
```

El panel pasa a usar `job:orderflow_error_ratio:5m`, que es una lectura directa.

La convención de nombre `nivel:metrica:operacion` no es obligatoria, pero permite
distinguir de un vistazo lo que es una métrica cruda de lo que es un cálculo.

## 4) Optimización en KQL / Kibana

- **Acota siempre el rango de tiempo.** `now-1h` es mucho más barato que "todo".
- Filtra por campos `keyword`, que están indexados:

  ```
  level: "ERROR" and event: "order_failed"
  ```

- Evita comodines al principio (`*error`): impiden usar el índice y obligan a
  recorrerlo entero.
- Para contar, usa agregaciones en vez de traer documentos y contarlos en Python.

## Ejercicio

1. Reescribe esta consulta agregando por `le` y compara el número de series antes
   y después con la función `n_series`:

   ```promql
   rate(orderflow_processing_duration_seconds_bucket[5m])
   ```

   *Ojo: al agregar un histograma hay que conservar la etiqueta `le`, o dejas de
   poder calcular percentiles.*

2. Propón una recording rule para el p95 de latencia. Escríbela completa, con su
   nombre siguiendo la convención, y di qué panel de tu dashboard la usaría.